
# HTML, CSS et inspection d'une page web

**Site étudié :** [Books to Scrape](https://books.toscrape.com/)  
**Catégorie sélectionnée :** Nonfiction

## Étape 1 — Périmètre de collecte

| Indicateur | Valeur |
|---|---:|
| Catégorie | Nonfiction |
| Nombre de livres annoncé | 110 |
| Nombre de pages | 6 |
| Nombre maximal de livres par page | 20 |

La catégorie contient 110 livres répartis sur 6 pages.

## Étape 2 — Identification des fiches produits

Chaque livre est représenté par un élément HTML possédant la classe `product_pod`.

**Sélecteur CSS :** `.product_pod`

**Commande de vérification :**

```javascript
document.querySelectorAll('.product_pod').length
```

**Résultats :**

| Page | Nombre de fiches |
|---|---:|
| 1 | 20 |
| 2 | 20 |
| 3 | 20 |
| 4 | 20 |
| 5 | 20 |
| 6 | 10 |
| **Total** | **110** |

**Contrôle de cohérence :**

20 + 20 + 20 + 20 + 20 + 10 = **110 livres**

Le nombre de fiches identifiées correspond au nombre de livres annoncé par le site.

## Étape 3 — Identification des données à extraire

| Donnée | Sélecteur CSS | Source de la valeur |
|---|---|---|
| Titre | `a[title]` | Attribut `title` |
| Prix | `p.price_color` | Contenu texte |
| Disponibilité | `.in_stock` | Contenu texte |
| Note | `.star-rating` | Attribut `class` |
| Lien vers la fiche | `a[href]` | Attribut `href` |

**Remarque :** les sélecteurs permettent d'identifier les éléments HTML. L'extraction de leurs valeurs nécessite ensuite de lire leur contenu texte ou l'attribut correspondant.

## Étape 4 — Vérification des sélecteurs

Les sélecteurs suivants ont été testés dans la console du navigateur, sur la première page de la catégorie.

**Nombre de fiches produits :**

```javascript
document.querySelectorAll('.product_pod').length
// Résultat : 20
```

**Nombre de prix :**

```javascript
document.querySelectorAll('article.product_pod p.price_color').length
// Résultat : 20
```

**Nombre de notes :**

```javascript
document.querySelectorAll('article.product_pod .star-rating').length
// Résultat : 20
```

### Bilan des contrôles

| Élément | Nombre obtenu | Résultat |
|---|---:|---|
| Fiches produits | 20 | Conforme |
| Prix | 20 | Conforme |
| Notes | 20 | Conforme |

Les trois sélecteurs retournent le même nombre d'éléments.

Les contrôles de cardinalité sont cohérents avec les 20 livres affichés sur cette page.

## Étape 5 — Pagination

**Sélecteur du lien vers la page suivante :** `li.next a`

**Valeur de l'attribut `href` sur la première page :**

```text
page-2.html
```

Le lien vers la page suivante est relatif à l'URL de la page courante.

**Condition d'arrêt :**

Lorsque le lien vers la page suivante est absent, le script doit arrêter la pagination : il a atteint la dernière page.

L'absence de ce lien constitue une condition normale de fin de collecte.

## Étape 6 — Vérification du code source

Le prix d'un livre est présent dans le code source HTML de la page.

**Conséquence :**

La donnée peut être extraite directement à partir du HTML téléchargé, sans nécessiter l'exécution de JavaScript.

## Étape 7 — Informations supplémentaires

La fiche détaillée d'un livre contient des informations absentes de la liste des produits.

Deux informations ont été identifiées :

| Information | Emplacement |
|---|---|
| Taxe (`Tax`) | Tableau des informations produit |
| Quantité en stock (`Availability`) | Tableau des informations produit |

La liste des produits indique uniquement la disponibilité du livre, tandis que la fiche détaillée permet de connaître la quantité en stock.

In [40]:
# Les imports pour le projet
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from urllib.parse import urljoin


In [24]:
# Constantes python
_URL = "https://books.toscrape.com/catalogue/category/books/nonfiction_13/page-1.html"
_ENTETES = {"User-Agent": "bootcamp-data/1.0 (Wild Code School)"}
_NOTES = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

In [38]:
def lire_une_fiche(fiche):
    dict_fiche = {}
    lien_titre = fiche.select_one("h3 a")
    rating = fiche.select_one("p.star-rating")["class"]

    dict_fiche['prix'] = fiche.select_one("p.price_color").get_text(strip=True)
    dict_fiche['titre'] = lien_titre["title"]
    dict_fiche['url'] = lien_titre["href"]
    dict_fiche['note'] = _NOTES[rating[1]]
    dict_fiche['dispo'] = fiche.select_one("p.instock").get_text(strip=True)

    return dict_fiche


def recuperer_une_page(url, encoding="utf-8"):
    r = requests.get(url, headers=_ENTETES, timeout=10)
    r.encoding = encoding

    if r.status_code != 200:
        raise ValueError(f"Server returned status {r.status_code} : break")
    soup = BeautifulSoup(r.text, "html.parser")
    fiches_livres = soup.select("article.product_pod")
    
    liste_fiche =[]
    for fiche in fiches_livres:
        liste_fiche.append(lire_une_fiche(fiche))

    df = pd.DataFrame(liste_fiche)

    suivant = soup.select_one("li.next a")
    url_suivant = urljoin(url, suivant["href"]) if suivant else None

    return df, url_suivant

In [26]:
print("Etape 2 :\n")
df_page = recuperer_une_page(_URL)
print(df_page)

Etape 2 :

      prix                                              titre  \
0   £40.30  Worlds Elsewhere: Journeys Around Shakespeare’...   
1   £31.05  The Five Love Languages: How to Express Heartf...   
2   £26.41                              Reasons to Stay Alive   
3   £23.11  #HigherSelfie: Wake Up Your Life. Free Your So...   
4   £44.18  Unseen City: The Majesty of Pigeons, the Discr...   
5   £31.12  Throwing Rocks at the Google Bus: How Growth B...   
6   £16.77  The Life-Changing Magic of Tidying Up: The Jap...   
7   £37.13  The Gutsy Girl: Escapades for Your Life of Epi...   
8   £56.06  The Electric Pencil: Drawings from Inside Stat...   
9   £41.83  Spark Joy: An Illustrated Master Class on the ...   
10  £19.83  Reskilling America: Learning to Labor in the T...   
11  £22.00          In the Country We Love: My Family Divided   
12  £54.35  Everydata: The Misinformation Hidden in the Li...   
13  £29.14  Call the Nurse: True Stories of a Country Nurs...   
14  £30.81  Al

In [ ]:
def collecter_toutes_pages(url_depart, pages_max):
    url = url_depart
    df_livres = None
    page = 1

    while url and page <= pages_max:

        df_recuperé, url = recuperer_une_page(url)
        df_livres = pd.concat([df_livres, df_recuperé], ignore_index=True)
        print(f"Page {page} : {df_recuperé.shape[0]} livres (total {df_livres.shape[0]})")

        page += 1
        time.sleep(1)
    
    return df_livres
        
        
df_livres = collecter_toutes_pages(_URL, 10)
print(f"{df_livres.shape[0]} (attendu 110) : {df_livres.shape[0] == 110}")

Page 1 : 20 livres (total 20)
Page 2 : 20 livres (total 40)
Page 3 : 20 livres (total 60)
Page 4 : 20 livres (total 80)
Page 5 : 20 livres (total 100)
Page 6 : 10 livres (total 110)
     prix                                              titre  \
0  £40.30  Worlds Elsewhere: Journeys Around Shakespeare’...   
1  £31.05  The Five Love Languages: How to Express Heartf...   
2  £26.41                              Reasons to Stay Alive   
3  £23.11  #HigherSelfie: Wake Up Your Life. Free Your So...   
4  £44.18  Unseen City: The Majesty of Pigeons, the Discr...   

                                                 url  note     dispo  
0  ../../../worlds-elsewhere-journeys-around-shak...     5  In stock  
1  ../../../the-five-love-languages-how-to-expres...     3  In stock  
2      ../../../reasons-to-stay-alive_959/index.html     2  In stock  
3  ../../../higherselfie-wake-up-your-life-free-y...     5  In stock  
4  ../../../unseen-city-the-majesty-of-pigeons-th...     4  In stock  
